# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# نافذة ديناميكية بناءً على المدى الزمني الفعلي (29 يوم متاحين -> half_window ≈ 14)
span = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           DATEDIFF('day', MIN(report_date), MAX(report_date)) AS total_days
    FROM {TABLES['fact_daily']}
""").df()
half_window = span['total_days'].iloc[0] // 2
print("half_window:", half_window)

FINAL_THRESHOLD = 100  # الأصلي شغال كويس بعد تصحيح الـ interval bug — 77,155 صف

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= {FINAL_THRESHOLD}
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

print(f'features: {len(features):,} rows | qsignals: {len(qsignals):,} rows')

data = features.merge(qsignals, on='content_hash_id', how='left')

volatility = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
    SELECT f.content_hash_id, STDDEV(f.gsc_avg_position) AS position_volatility
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
    GROUP BY 1
""").df()
data = data.merge(volatility, on='content_hash_id', how='left')

data['top_query_share'] = data['top_query_impressions'] / data['kept_impressions']

fill_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']
data[fill_cols] = data[fill_cols].fillna(0)

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share',
                 'anon_share', 'top_query_share', 'position_volatility']

print("data rows:", len(data))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

half_window: 14


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features: 77,155 rows | qsignals: 133,852 rows
data rows: 77155


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,position_volatility,top_query_share,is_declining
0,client_62f4a7e64f5e0096,content_061bd6db307f182d,283.0,365.0,1.0,30.044272,9.0,0.111457,0.750934,50.0,221.0,13.672740,0.226244,1
1,client_62f4a7e64f5e0096,content_84a6cccf3766c488,433.0,282.0,1.0,11.000782,16.0,0.130712,0.716525,304.0,575.0,10.957168,0.528696,0
2,client_62f4a7e64f5e0096,content_d3450afad809b945,112.0,111.0,1.0,36.846808,6.0,0.335548,0.438538,34.0,136.0,24.123386,0.250000,0
3,client_62f4a7e64f5e0096,content_8c65803b7593f062,719.0,1315.0,1.0,10.154206,35.0,0.022568,0.696970,453.0,2647.0,2.767153,0.171137,1
4,client_62f4a7e64f5e0096,content_ebf6830b74454a5e,668.0,881.0,8.0,3.631036,6.0,0.021470,0.918891,287.0,400.0,0.999904,0.717500,1


In [6]:
import pandas as pd

# افحصي الأعمدة الرقمية الأساسية عندك (عدّلي القايمة حسب اللي متاح فعليًا في dataset الجديد)
cols_to_check = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share',
                  'top_query_share', 'position_volatility']

for col in cols_to_check:
    print(f"--- {col} ---")
    print(data[col].describe(percentiles=[.25, .5, .75, .9, .95, .99]))
    print()

--- imp_prev30 ---
count     77155.000000
mean       1323.910479
std        4192.900377
min         100.000000
25%         192.000000
50%         406.000000
75%        1077.000000
90%        2822.000000
95%        5036.000000
99%       14518.920000
max      425215.000000
Name: imp_prev30, dtype: float64

--- visible_queries ---
count    77155.000000
mean        28.293656
std         59.734115
min          0.000000
25%          6.000000
50%         14.000000
75%         31.000000
90%         65.000000
95%        101.000000
99%        230.000000
max       7889.000000
Name: visible_queries, dtype: float64

--- rare_share ---
count    77155.000000
mean         0.095991
std          0.096537
min          0.000000
25%          0.029200
50%          0.064908
75%          0.130221
90%          0.224113
95%          0.297333
99%          0.442435
max          0.908705
Name: rare_share, dtype: float64

--- anon_share ---
count    77155.000000
mean         0.658656
std          0.237756
min      

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [7]:

signal_col = 'position_volatility'

test = data.groupby(pd.qcut(data[signal_col], 4, duplicates='drop'))['is_declining'].mean()
print(test)

position_volatility
(0.014599999999999998, 2.143]    0.374410
(2.143, 5.157]                   0.497486
(5.157, 9.901]                   0.523901
(9.901, 109.903]                 0.597594
Name: is_declining, dtype: float64


/tmp/ipykernel_2240/378952151.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = data.groupby(pd.qcut(data[signal_col], 4, duplicates='drop'))['is_declining'].mean()


### Signal test #2 — rare_share
Hypothesis: pages that depend heavily on rare/long-tail queries (high rare_share)
are more exposed to decline, since long-tail traffic is less stable than head terms.

In [9]:
signal_col = 'rare_share'
test2 = data.groupby(pd.qcut(data[signal_col], 4, duplicates='drop'), observed=True)['is_declining'].mean()
print(test2)

rare_share
(-0.001, 0.0292]    0.521385
(0.0292, 0.0649]    0.556327
(0.0649, 0.13]      0.498756
(0.13, 0.909]       0.416922
Name: is_declining, dtype: float64


**Signal test #2 — rare_share:** MIXED. Decline rate does not move consistently with
rare_share — it rises slightly then falls, ending at 41.7% in the highest quartile
versus 55.6% in the second. The direction in the top quartile actually runs opposite
to the hypothesis, but the pattern isn't clean enough across all four quartiles to
call it a reversal outright.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:

flag_col = 'top_query_share'
threshold = 0.5

data['flag_triggered'] = data[flag_col] > threshold
print(data.groupby('flag_triggered')['is_declining'].mean())
print(data['flag_triggered'].value_counts())

flag_triggered
False    0.501202
True     0.485832
Name: is_declining, dtype: float64
flag_triggered
False    62827
True     14328
Name: count, dtype: int64


### Signal test #3 — visible_queries
Hypothesis: pages ranking for more distinct queries (higher visible_queries) are
more resilient and less likely to decline, since their traffic isn't concentrated
in one place.

In [10]:
signal_col = 'visible_queries'
test3 = data.groupby(pd.qcut(data[signal_col], 4, duplicates='drop'), observed=True)['is_declining'].mean()
print(test3)

visible_queries
(-0.001, 6.0]     0.484108
(6.0, 14.0]       0.499014
(14.0, 31.0]      0.490649
(31.0, 7889.0]    0.521934
Name: is_declining, dtype: float64


**Signal test #3 — visible_queries:** FALSE. Decline rate stays within a narrow
48–52% band across all quartiles, with the highest-diversity quartile showing the
highest (not lowest) decline rate. The data does not support query diversity as a
protective signal.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

Position volatility is the one signal worth acting on: content teams should
prioritize monitoring pages with high week-to-week ranking swings, since decline
risk climbs steadily and consistently with volatility across the full range.

The other two candidate signals don't hold up. Query diversity (visible_queries)
shows no meaningful relationship with decline, and rare_share moves inconsistently
rather than in the expected direction — neither should be used to triage at-risk
pages. The top_query_share flag specifically should be reviewed or retired, since
flagged and unflagged pages decline at essentially the same rate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.